<a href="https://colab.research.google.com/github/khanhlt2185/rag_bot/blob/main/LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import json

def process_constitution(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as f:
        text = f.read()

    chunks = re.split(r'\n(?=Điều \d+)', text)

    documents = []

    for chunk in chunks:
        chunk = chunk.strip()
        if not chunk:
            continue

        parts = chunk.split('\n', 1)
        title = parts[0].strip()
        content = parts[1].strip() if len(parts) > 1 else ""

        documents.append({
            "title": title,
            "content": content
        })

    with open(output_file, 'w', encoding='utf-8') as out_file:
        json.dump(documents, out_file, ensure_ascii=False, indent=4)

process_constitution("hien-phap-2013.txt", "hien_phap_2013_processed.json")

In [1]:
!pip install chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 96.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found 

In [2]:
import json
import chromadb
from chromadb.utils import embedding_functions

print("khởi tạo Database")

client = chromadb.PersistentClient(path="/content/chroma_db")

sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="paraphrase-multilingual-MiniLM-L12-v2")

collection = client.get_or_create_collection(
    name="hien_phap_vn",
    embedding_function=sentence_transformer_ef
)

with open('/content/hien_phap_2013_processed.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

docs = []
metadatas = []
ids = []

for i, item in enumerate(data):
    docs.append(item["content"])
    metadatas.append({"title": item["title"]})
    ids.append(f"dieu_{i+1}")

print(f"....")
collection.add(
    documents=docs,
    metadatas=metadatas,
    ids=ids
)

print("ok")

khởi tạo Database


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

....
ok


In [4]:
print("🔍 Đang tìm kiếm trong cơ sở dữ liệu...\n")

# 1. Đặt một câu hỏi thực tế
cau_hoi = "Những ai có quyền bầu cử và ứng cử đại biểu Quốc hội?"

# 2. Truy xuất 2 Điều luật có nội dung sát với câu hỏi nhất
ket_qua = collection.query(
    query_texts=[cau_hoi],
    n_results=5
)

# 3. In kết quả ra màn hình
print(f"❓ Câu hỏi của người dùng: '{cau_hoi}'\n")
print("✅ Các Điều luật liên quan nhất tìm được:\n")

for i in range(len(ket_qua['documents'][0])):
    tieu_de = ket_qua['metadatas'][0][i]['title']
    noi_dung = ket_qua['documents'][0][i]

    print(f"--- {tieu_de} ---")
    print(noi_dung)
    print("-" * 50)

🔍 Đang tìm kiếm trong cơ sở dữ liệu...

❓ Câu hỏi của người dùng: 'Những ai có quyền bầu cử và ứng cử đại biểu Quốc hội?'

✅ Các Điều luật liên quan nhất tìm được:

--- Điều 79 ---
1. Đại biểu Quốc hội là người đại diện cho ý chí, nguyện vọng của Nhân dân ở đơn vị bầu cử ra mình và của Nhân dân cả nước.

2. Đại biểu Quốc hội liên hệ chặt chẽ với cử tri, chịu sự giám sát của cử tri; thu thập và phản ánh trung thực ý kiến, nguyện vọng của cử tri với Quốc hội, các cơ quan, tổ chức hữu quan; thực hiện chế độ tiếp xúc và báo cáo với cử tri về hoạt động của đại biểu và của Quốc hội; trả lời yêu cầu và kiến nghị của cử tri; theo dõi, đôn đốc việc giải quyết khiếu nại, tố cáo và hướng dẫn, giúp đỡ việc thực hiện quyền khiếu nại, tố cáo.

3. Đại biểu Quốc hội phổ biến và vận động Nhân dân thực hiện Hiến pháp và pháp luật.
--------------------------------------------------
--- Điều 7 ---
1. Việc bầu cử đại biểu Quốc hội và đại biểu Hội đồng nhân dân được tiến hành theo nguyên tắc phổ thông, bình